In [ ]:
import tensorflow as tf
import numpy as np
from ten sorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

In [2]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

In [3]:
train_set = train_datagen.flow_from_directory(
    r"E:\Plant_Diseases\train",
    target_size=(150,150),
    batch_size=32,
    class_mode='categorical'
)

validation_set = val_datagen.flow_from_directory(
    r"E:\Plant_Diseases\valid",
    target_size=(150,150),
    batch_size=32,
    class_mode='categorical'
)

Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.


In [4]:
labels = train_set.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(0.917593462823726), 1: np.float64(0.9309856170370566), 2: np.float64(1.0510616028708133), 3: np.float64(0.9212492136716293), 4: np.float64(1.0186500115928587), 5: np.float64(1.0991493886230728), 6: np.float64(1.013071424453796), 7: np.float64(1.1265946535034297), 8: np.float64(0.9700411227334198), 9: np.float64(0.9695327154363897), 10: np.float64(0.9950879080433737), 11: np.float64(0.9798031891168599), 12: np.float64(0.9634731359649122), 13: np.float64(1.074255761354606), 14: np.float64(1.0933028493218864), 15: np.float64(0.9203325477873789), 16: np.float64(1.006457247580322), 17: np.float64(1.0705257066276803), 18: np.float64(0.9669986518832365), 19: np.float64(0.9305173144127925), 20: np.float64(0.9540321923943432), 21: np.float64(0.9540321923943432), 22: np.float64(1.0141822483841183), 23: np.float64(1.038668400366441), 24: np.float64(0.9148706335571868), 25: np.float64(1.065592408440456), 26: np.float64(1.0427668664332759), 27: np.float64(1.0141822483841183), 28: np.

In [5]:
cnn = tf.keras.models.Sequential()

In [6]:
# Block 1
cnn.add(tf.keras.layers.Conv2D(32,3,activation='relu',input_shape=(150,150,3)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(2,2))

C:\Users\soot\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
# Block 2
cnn.add(tf.keras.layers.Conv2D(64,3,activation='relu'))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(2,2))

In [8]:
# Block 3
cnn.add(tf.keras.layers.Conv2D(128,3,activation='relu'))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(2,2))

In [9]:

# 🔥 Block إضافي (جديد)
cnn.add(tf.keras.layers.Conv2D(256,3,activation='relu'))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(2,2))

In [10]:
cnn.add(tf.keras.layers.Flatten())

cnn.add(tf.keras.layers.Dense(256,activation='relu'))
cnn.add(tf.keras.layers.Dropout(0.5))

cnn.add(tf.keras.layers.Dense(len(train_set.class_indices),activation='softmax'))

In [11]:
cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
# وقف ذكي
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)


In [13]:
# تقليل learning rate
lr_reduce = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [14]:
# حفظ أفضل موديل
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

In [15]:
history = cnn.fit(
    train_set,
    validation_data=validation_set,
    epochs=100,
    callbacks=[early_stop, lr_reduce, checkpoint],
    class_weight=class_weights
)

Epoch 1/100
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4468 - loss: 2.0924
Epoch 1: val_accuracy improved from None to 0.66054, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 2461s 1s/step - accuracy: 0.5840 - loss: 1.4958 - val_accuracy: 0.6605 - val_loss: 1.2872 - learning_rate: 1.0000e-04
Epoch 2/100
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 837ms/step - accuracy: 0.7628 - loss: 0.8094
Epoch 2: val_accuracy improved from 0.66054 to 0.72951, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 1976s 899ms/step - accuracy: 0.7863 - loss: 0.7239 - val_accuracy: 0.7295 - val_loss: 1.0615 - learning_rate: 1.0000e-04
Epoch 3/100
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 841ms/step - accuracy: 0.8363 - loss: 0.5396
Epoch 3: val_accuracy improved from 0.72951 to 0.80981, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
2197/2197

In [16]:
cnn.save("cnn_final100.keras")